In [1]:
# ============================================================
# 12_AURORA_paper_tables_figures_and_manuscript_assets.ipynb
# AURORA-TWETF Paper Tables, Figures, and Manuscript Assets
#
# Purpose:
# 1. Consolidate final results from Notebooks 08B, 09, 10, and 11.
# 2. Produce paper-ready tables, figures, captions, and manuscript text.
# 3. Create a final evidence-based result narrative.
# 4. Avoid unsupported claims:
#    - Do NOT claim total-return superiority.
#    - Do NOT claim statistically significant Sharpe superiority.
#    - DO claim statistically significant drawdown reduction.
#
# Educational/research use only.
# Not personalized financial advice.
# ============================================================

from __future__ import annotations

import json
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. Paths and run IDs
# ============================================================

PROJECT_CODE = "AURORA_TWETF"
PUBLICATION_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")

OUTPUT_ROOT = PUBLICATION_ROOT / "outputs" / PROJECT_CODE
TABLE_DIR = OUTPUT_ROOT / "tables"
REPORT_DIR = OUTPUT_ROOT / "reports"
FIGURE_DIR = OUTPUT_ROOT / "figures"

NOTEBOOK08B_RUN_ID = "20260624_070827"
NOTEBOOK09_RUN_ID = "20260624_072914"
NOTEBOOK10_RUN_ID = "20260624_100748"
NOTEBOOK11_RUN_ID = "20260624_124834"

NOTEBOOK08B_ROOT = OUTPUT_ROOT / "aligned_oos_reanalysis" / f"run_{NOTEBOOK08B_RUN_ID}"
NOTEBOOK09_ROOT = OUTPUT_ROOT / "validation_optimized_regime_templates" / f"run_{NOTEBOOK09_RUN_ID}"
NOTEBOOK10_ROOT = OUTPUT_ROOT / "uncertainty_aware_mean_variance_allocation" / f"run_{NOTEBOOK10_RUN_ID}"
NOTEBOOK11_ROOT = OUTPUT_ROOT / "statistical_significance_block_bootstrap" / f"run_{NOTEBOOK11_RUN_ID}"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / "paper_tables_figures_manuscript_assets" / f"run_{RUN_ID}"

TABLE_RUN_DIR = RUN_ROOT / "tables"
PLOT_DIR = RUN_ROOT / "plots"
PAPER_FIGURE_DIR = RUN_ROOT / "paper_figures"
MANUSCRIPT_DIR = RUN_ROOT / "manuscript_assets"
LATEX_DIR = RUN_ROOT / "latex_tables"
REPORT_RUN_DIR = RUN_ROOT / "reports"

for d in [
    OUTPUT_ROOT,
    TABLE_DIR,
    REPORT_DIR,
    FIGURE_DIR,
    RUN_ROOT,
    TABLE_RUN_DIR,
    PLOT_DIR,
    PAPER_FIGURE_DIR,
    MANUSCRIPT_DIR,
    LATEX_DIR,
    REPORT_RUN_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 80)
print("AURORA-TWETF Notebook 12: Paper Tables, Figures, and Manuscript Assets")
print("=" * 80)
print("Timestamp UTC :", RUN_TIMESTAMP)
print("Run ID        :", RUN_ID)
print("Notebook 08B  :", NOTEBOOK08B_ROOT)
print("Notebook 09   :", NOTEBOOK09_ROOT)
print("Notebook 10   :", NOTEBOOK10_ROOT)
print("Notebook 11   :", NOTEBOOK11_ROOT)
print("Run root      :", RUN_ROOT)
print("=" * 80)

# ============================================================
# 2. Required input files
# ============================================================

INPUTS = {
    "08B_aligned_oos_rankings": TABLE_DIR / f"table_69_aligned_oos_rankings_{NOTEBOOK08B_RUN_ID}.csv",
    "08B_aligned_test_rankings": TABLE_DIR / f"table_70_aligned_test_only_rankings_{NOTEBOOK08B_RUN_ID}.csv",
    "08B_aurora_vs_benchmark": TABLE_DIR / f"table_71_aligned_aurora_vs_best_benchmark_{NOTEBOOK08B_RUN_ID}.csv",
    "09_optimized_rankings": TABLE_DIR / f"table_82_comparison_rankings_optimized_vs_benchmarks_{NOTEBOOK09_RUN_ID}.csv",
    "09_aurora_vs_benchmark": TABLE_DIR / f"table_83_optimized_aurora_vs_best_benchmark_{NOTEBOOK09_RUN_ID}.csv",
    "10_uamv_rankings": TABLE_DIR / f"table_93_comparison_rankings_uamv_vs_benchmarks_{NOTEBOOK10_RUN_ID}.csv",
    "10_uamv_vs_benchmark": TABLE_DIR / f"table_94_uamv_aurora_vs_best_benchmark_{NOTEBOOK10_RUN_ID}.csv",
    "11_observed_pairs": TABLE_DIR / f"table_97_observed_paired_performance_differences_{NOTEBOOK11_RUN_ID}.csv",
    "11_hac_tests": TABLE_DIR / f"table_98_hac_newey_west_mean_excess_return_tests_{NOTEBOOK11_RUN_ID}.csv",
    "11_bootstrap_primary": TABLE_DIR / f"table_100_block_bootstrap_summary_primary_block_{NOTEBOOK11_RUN_ID}.csv",
    "11_decision_table": TABLE_DIR / f"table_102_paper_statistical_decision_table_{NOTEBOOK11_RUN_ID}.csv",
    "11_diagnostic_summary": TABLE_DIR / f"table_105_notebook11_diagnostic_summary_{NOTEBOOK11_RUN_ID}.csv",
    "11_equity_drawdown": TABLE_DIR / f"table_104_equity_and_drawdown_diagnostics_{NOTEBOOK11_RUN_ID}.csv",
    "11_rolling": TABLE_DIR / f"table_103_rolling_performance_diagnostics_{NOTEBOOK11_RUN_ID}.csv",
}

missing = [name for name, path in INPUTS.items() if not path.exists()]
if missing:
    print("Missing input files:")
    for name in missing:
        print(name, INPUTS[name])
    raise FileNotFoundError("Some required input files are missing.")

# ============================================================
# 3. Utility functions
# ============================================================

def save_json(path, obj):
    Path(path).write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def sha256_file(path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)

    return h.hexdigest()

def make_file_manifest(root):
    root = Path(root)
    rows = []

    for p in sorted(root.rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })

    return pd.DataFrame(rows)

def safe_name(x):
    return (
        str(x)
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(" ", "_")
        .replace(".", "_")
        .replace("%", "pct")
    )

def read_csv(path):
    return pd.read_csv(path)

def write_markdown(path, text):
    Path(path).write_text(text.strip() + "\n", encoding="utf-8")

def format_float(x, digits=4):
    if pd.isna(x):
        return ""
    return f"{float(x):.{digits}f}"

def copy_to_global_table(local_path, global_name):
    local_path = Path(local_path)
    global_path = TABLE_DIR / global_name
    global_path.write_bytes(local_path.read_bytes())
    return global_path

def save_table(df, local_name, global_table_number, description):
    local_csv = TABLE_RUN_DIR / local_name
    global_csv = TABLE_DIR / f"table_{global_table_number}_{description}_{RUN_ID}.csv"

    df.to_csv(local_csv, index=False)
    df.to_csv(global_csv, index=False)

    return local_csv, global_csv

def save_latex_table(df, filename, caption, label, float_format="%.4f"):
    latex_path = LATEX_DIR / filename

    latex = df.to_latex(
        index=False,
        escape=False,
        float_format=lambda x: float_format % x if pd.notnull(x) else "",
        caption=caption,
        label=label,
    )

    latex_path.write_text(latex, encoding="utf-8")
    return latex_path

def policy_short_name(policy_name):
    mapping = {
        "AURORA10_UAMV_B_more60_defensive": "AURORA10-UAMV-B",
        "AURORA10_UAMV_D_low_turnover": "AURORA10-UAMV-D",
        "AURORA10_UAMV_E_no_regime_tilt_control": "AURORA10-UAMV-E",
        "AURORA10_UAMV_A_balanced": "AURORA10-UAMV-A",
        "AURORA10_UAMV_C_more60_growth": "AURORA10-UAMV-C",
        "AURORA10_validation_selected_UAMV": "AURORA10 Validation-Selected",
        "B6_00881_only": "00881-only",
        "B3_0050_only": "0050-only",
        "B1_equal_weight_all_etfs": "Equal-weight ETFs",
        "B10_momentum_top2_63d": "Momentum top-2",
        "B16_minimum_variance_126d": "Minimum variance",
        "B4_006208_only": "006208-only",
        "AURORA_P3_purged_wf_no_cash_boost": "AURORA P3",
        "AURORA_P0_purged_wf_baseline_60_40": "AURORA P0",
        "AURORA09_VOT_A_baseline_60_40_cash20": "AURORA09 VOT-A",
    }
    return mapping.get(policy_name, policy_name)

# ============================================================
# 4. Load all result tables
# ============================================================

print("\n" + "=" * 80)
print("Step 1: Loading result tables")
print("=" * 80)

aligned_oos_08b = read_csv(INPUTS["08B_aligned_oos_rankings"])
aligned_test_08b = read_csv(INPUTS["08B_aligned_test_rankings"])
aurora_vs_08b = read_csv(INPUTS["08B_aurora_vs_benchmark"])

optimized_rank_09 = read_csv(INPUTS["09_optimized_rankings"])
aurora_vs_09 = read_csv(INPUTS["09_aurora_vs_benchmark"])

uamv_rank_10 = read_csv(INPUTS["10_uamv_rankings"])
uamv_vs_10 = read_csv(INPUTS["10_uamv_vs_benchmark"])

observed_pairs_11 = read_csv(INPUTS["11_observed_pairs"])
hac_11 = read_csv(INPUTS["11_hac_tests"])
bootstrap_11 = read_csv(INPUTS["11_bootstrap_primary"])
decision_11 = read_csv(INPUTS["11_decision_table"])
diagnostic_11 = read_csv(INPUTS["11_diagnostic_summary"])

equity_drawdown_11 = read_csv(INPUTS["11_equity_drawdown"])
rolling_11 = read_csv(INPUTS["11_rolling"])

equity_drawdown_11["date"] = pd.to_datetime(equity_drawdown_11["date"])
rolling_11["date"] = pd.to_datetime(rolling_11["date"])

print("Notebook 08B aligned test rankings:", aligned_test_08b.shape)
print("Notebook 09 optimized rankings     :", optimized_rank_09.shape)
print("Notebook 10 UAMV rankings          :", uamv_rank_10.shape)
print("Notebook 11 decision table         :", decision_11.shape)

# ============================================================
# 5. Define final paper policies
# ============================================================

PRIMARY_AURORA = "AURORA10_UAMV_B_more60_defensive"

PRIMARY_BENCHMARKS = [
    "B6_00881_only",
    "B3_0050_only",
    "B1_equal_weight_all_etfs",
    "B10_momentum_top2_63d",
    "B16_minimum_variance_126d",
]

FINAL_POLICY_SET = [
    PRIMARY_AURORA,
    "AURORA10_UAMV_D_low_turnover",
    "AURORA10_UAMV_E_no_regime_tilt_control",
    "AURORA10_UAMV_A_balanced",
    "AURORA10_UAMV_C_more60_growth",
    "AURORA10_validation_selected_UAMV",
] + PRIMARY_BENCHMARKS

# ============================================================
# 6. Create final performance table
# ============================================================

print("\n" + "=" * 80)
print("Step 2: Creating final performance table")
print("=" * 80)

final_perf = uamv_rank_10[uamv_rank_10["policy_name"].isin(FINAL_POLICY_SET)].copy()
final_perf["policy_short"] = final_perf["policy_name"].map(policy_short_name)

ordered = [p for p in FINAL_POLICY_SET if p in final_perf["policy_name"].values]
final_perf["policy_order"] = final_perf["policy_name"].apply(lambda x: ordered.index(x) if x in ordered else 999)
final_perf = final_perf.sort_values("policy_order")

final_perf_table = final_perf[
    [
        "policy_short",
        "policy_name",
        "policy_type",
        "n_days",
        "total_return",
        "annual_return",
        "annual_volatility",
        "sharpe_ratio",
        "sortino_ratio",
        "max_drawdown",
        "calmar_ratio",
        "allocation_composite_rank",
    ]
].copy()

final_perf_table = final_perf_table.rename(columns={
    "policy_short": "Policy",
    "policy_name": "Full policy name",
    "policy_type": "Policy type",
    "n_days": "Days",
    "total_return": "Total return",
    "annual_return": "Annual return",
    "annual_volatility": "Annual volatility",
    "sharpe_ratio": "Sharpe",
    "sortino_ratio": "Sortino",
    "max_drawdown": "Max drawdown",
    "calmar_ratio": "Calmar",
    "allocation_composite_rank": "Composite rank",
})

save_table(
    final_perf_table,
    local_name="paper_final_performance_table.csv",
    global_table_number=106,
    description="paper_final_performance_table",
)

save_latex_table(
    final_perf_table,
    filename="table_final_performance.tex",
    caption="Aligned strict-test performance of AURORA10-UAMV policies and primary benchmarks.",
    label="tab:final_performance",
)

print(final_perf_table.to_string(index=False))

# ============================================================
# 7. Create final statistical inference table
# ============================================================

print("\n" + "=" * 80)
print("Step 3: Creating statistical inference table")
print("=" * 80)

paper_metrics = [
    "diff_total_return",
    "diff_sharpe",
    "diff_sortino",
    "drawdown_improvement",
    "diff_calmar",
    "annualized_mean_excess_return",
]

stat_table = decision_11[
    (decision_11["aurora_policy"] == PRIMARY_AURORA)
    & (decision_11["benchmark_policy"].isin(PRIMARY_BENCHMARKS))
    & (decision_11["metric"].isin(paper_metrics))
].copy()

stat_table["AURORA"] = stat_table["aurora_policy"].map(policy_short_name)
stat_table["Benchmark"] = stat_table["benchmark_policy"].map(policy_short_name)

metric_labels = {
    "diff_total_return": "Total return difference",
    "diff_sharpe": "Sharpe difference",
    "diff_sortino": "Sortino difference",
    "drawdown_improvement": "Max drawdown improvement",
    "diff_calmar": "Calmar difference",
    "annualized_mean_excess_return": "Annualized mean excess return",
}

stat_table["Metric"] = stat_table["metric"].map(metric_labels)

stat_table_out = stat_table[
    [
        "AURORA",
        "Benchmark",
        "Metric",
        "observed_value",
        "ci95_lower",
        "ci95_upper",
        "probability_positive",
        "conclusion",
    ]
].copy()

stat_table_out = stat_table_out.rename(columns={
    "observed_value": "Observed",
    "ci95_lower": "CI95 lower",
    "ci95_upper": "CI95 upper",
    "probability_positive": "Bootstrap Pr(positive)",
    "conclusion": "Conclusion",
})

save_table(
    stat_table_out,
    local_name="paper_statistical_inference_table.csv",
    global_table_number=107,
    description="paper_statistical_inference_table",
)

save_latex_table(
    stat_table_out,
    filename="table_statistical_inference.tex",
    caption="Paired block-bootstrap inference for AURORA10-UAMV-B relative to primary benchmarks.",
    label="tab:statistical_inference",
)

print(stat_table_out.to_string(index=False))

# ============================================================
# 8. Create methodology progression table
# ============================================================

print("\n" + "=" * 80)
print("Step 4: Creating methodology progression table")
print("=" * 80)

best_08b_aurora = aligned_test_08b[
    aligned_test_08b["policy_type"].astype(str).str.contains("AURORA", case=False, na=False)
].sort_values("allocation_composite_rank").iloc[0]

best_08b_benchmark = aligned_test_08b[
    aligned_test_08b["policy_type"].isin(["passive_benchmark", "dynamic_financial_baseline"])
].sort_values("allocation_composite_rank").iloc[0]

best_09_aurora = optimized_rank_09[
    optimized_rank_09["policy_type"].astype(str).str.contains("AURORA", case=False, na=False)
].sort_values("allocation_composite_rank").iloc[0]

best_09_benchmark = optimized_rank_09[
    optimized_rank_09["policy_type"].isin(["passive_benchmark", "dynamic_financial_baseline"])
].sort_values("allocation_composite_rank").iloc[0]

best_10_aurora = uamv_rank_10[
    uamv_rank_10["policy_type"].astype(str).str.contains("AURORA", case=False, na=False)
].sort_values("allocation_composite_rank").iloc[0]

best_10_benchmark = uamv_rank_10[
    uamv_rank_10["policy_type"].isin(["passive_benchmark", "dynamic_financial_baseline"])
].sort_values("allocation_composite_rank").iloc[0]

progress_rows = [
    {
        "Stage": "Notebook 08B",
        "Allocation layer": "Hand-crafted regime templates, aligned dates",
        "Best AURORA policy": policy_short_name(best_08b_aurora["policy_name"]),
        "Best benchmark": policy_short_name(best_08b_benchmark["policy_name"]),
        "AURORA total return": best_08b_aurora["total_return"],
        "Benchmark total return": best_08b_benchmark["total_return"],
        "AURORA Sharpe": best_08b_aurora["sharpe_ratio"],
        "Benchmark Sharpe": best_08b_benchmark["sharpe_ratio"],
        "AURORA max drawdown": best_08b_aurora["max_drawdown"],
        "Benchmark max drawdown": best_08b_benchmark["max_drawdown"],
        "Conclusion": "Benchmark remained stronger after date alignment.",
    },
    {
        "Stage": "Notebook 09",
        "Allocation layer": "Validation-optimized random regime templates",
        "Best AURORA policy": policy_short_name(best_09_aurora["policy_name"]),
        "Best benchmark": policy_short_name(best_09_benchmark["policy_name"]),
        "AURORA total return": best_09_aurora["total_return"],
        "Benchmark total return": best_09_benchmark["total_return"],
        "AURORA Sharpe": best_09_aurora["sharpe_ratio"],
        "Benchmark Sharpe": best_09_benchmark["sharpe_ratio"],
        "AURORA max drawdown": best_09_aurora["max_drawdown"],
        "Benchmark max drawdown": best_09_benchmark["max_drawdown"],
        "Conclusion": "Validation template search overfit and underperformed.",
    },
    {
        "Stage": "Notebook 10",
        "Allocation layer": "Uncertainty-aware mean-variance allocation",
        "Best AURORA policy": policy_short_name(best_10_aurora["policy_name"]),
        "Best benchmark": policy_short_name(best_10_benchmark["policy_name"]),
        "AURORA total return": best_10_aurora["total_return"],
        "Benchmark total return": best_10_benchmark["total_return"],
        "AURORA Sharpe": best_10_aurora["sharpe_ratio"],
        "Benchmark Sharpe": best_10_benchmark["sharpe_ratio"],
        "AURORA max drawdown": best_10_aurora["max_drawdown"],
        "Benchmark max drawdown": best_10_benchmark["max_drawdown"],
        "Conclusion": "UAMV improved risk-adjusted rank and drawdown, while sacrificing return.",
    },
    {
        "Stage": "Notebook 11",
        "Allocation layer": "Block-bootstrap statistical inference",
        "Best AURORA policy": "AURORA10-UAMV-B",
        "Best benchmark": "00881-only",
        "AURORA total return": np.nan,
        "Benchmark total return": np.nan,
        "AURORA Sharpe": np.nan,
        "Benchmark Sharpe": np.nan,
        "AURORA max drawdown": np.nan,
        "Benchmark max drawdown": np.nan,
        "Conclusion": "Drawdown reduction was statistically supported; Sharpe improvement was not.",
    },
]

progression_table = pd.DataFrame(progress_rows)

save_table(
    progression_table,
    local_name="paper_methodology_progression_table.csv",
    global_table_number=108,
    description="paper_methodology_progression_table",
)

save_latex_table(
    progression_table,
    filename="table_methodology_progression.tex",
    caption="Evolution of the AURORA-TWETF allocation layer across experiments.",
    label="tab:methodology_progression",
)

print(progression_table.to_string(index=False))

# ============================================================
# 9. Create AURORA10 configuration table
# ============================================================

print("\n" + "=" * 80)
print("Step 5: Creating AURORA10-UAMV-B configuration table")
print("=" * 80)

uamv_b_config = [
    ("Model family", "Uncertainty-Aware Mean-Variance"),
    ("Policy name", "AURORA10_UAMV_B_more60_defensive"),
    ("20-day probability weight", "0.30"),
    ("60-day probability weight", "0.70"),
    ("Expected-return lookback", "63 trading days"),
    ("Covariance lookback", "126 trading days"),
    ("Mean shrinkage to zero", "0.60"),
    ("Momentum weight", "0.40"),
    ("Base risk aversion", "10.0"),
    ("Uncertainty risk multiplier", "2.5"),
    ("Bearish risk multiplier", "2.0"),
    ("Turnover penalty", "0.25"),
    ("Regime tilt strength", "0.25"),
    ("Maximum ETF weight", "0.45"),
    ("Maximum 00881 weight", "0.30"),
    ("Maximum cash weight", "0.60"),
    ("Interpretation", "Defensive UAMV configuration emphasizing 60-day regime uncertainty."),
]

config_table = pd.DataFrame(uamv_b_config, columns=["Parameter", "Value"])

save_table(
    config_table,
    local_name="paper_aurora10_uamv_b_configuration.csv",
    global_table_number=109,
    description="paper_aurora10_uamv_b_configuration",
)

save_latex_table(
    config_table,
    filename="table_uamv_b_configuration.tex",
    caption="Configuration of the primary AURORA10-UAMV-B allocation policy.",
    label="tab:uamv_b_config",
)

print(config_table.to_string(index=False))

# ============================================================
# 10. Create paper figures
# ============================================================

print("\n" + "=" * 80)
print("Step 6: Creating paper figures")
print("=" * 80)

FIGURE_POLICIES = [
    PRIMARY_AURORA,
    "B6_00881_only",
    "B3_0050_only",
    "B1_equal_weight_all_etfs",
    "B10_momentum_top2_63d",
]

eq_fig_df = equity_drawdown_11[equity_drawdown_11["policy_name"].isin(FIGURE_POLICIES)].copy()
eq_fig_df["Policy"] = eq_fig_df["policy_name"].map(policy_short_name)

plt.figure(figsize=(12, 6))
for policy, grp in eq_fig_df.groupby("Policy"):
    grp = grp.sort_values("date")
    plt.plot(grp["date"], grp["equity"], linewidth=1.8, label=policy)

plt.title("Aligned strict-test equity curves")
plt.xlabel("Date")
plt.ylabel("Equity, initial capital = 1")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
equity_fig = PLOT_DIR / "figure_final_equity_curves.png"
plt.savefig(equity_fig, dpi=240)
plt.close()

plt.figure(figsize=(12, 6))
for policy, grp in eq_fig_df.groupby("Policy"):
    grp = grp.sort_values("date")
    plt.plot(grp["date"], grp["drawdown"], linewidth=1.6, label=policy)

plt.title("Aligned strict-test drawdown curves")
plt.xlabel("Date")
plt.ylabel("Drawdown")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
drawdown_fig = PLOT_DIR / "figure_final_drawdowns.png"
plt.savefig(drawdown_fig, dpi=240)
plt.close()

perf_plot_df = final_perf_table.copy()

plt.figure(figsize=(10, 6))
plot_df = perf_plot_df.sort_values("Sharpe", ascending=False)
sns.barplot(data=plot_df, y="Policy", x="Sharpe", color="#4C72B0")
plt.title("Aligned strict-test Sharpe ratio")
plt.xlabel("Sharpe ratio")
plt.ylabel("")
plt.tight_layout()
sharpe_fig = PLOT_DIR / "figure_final_sharpe_bar.png"
plt.savefig(sharpe_fig, dpi=240)
plt.close()

plt.figure(figsize=(10, 6))
plot_df = perf_plot_df.sort_values("Max drawdown", ascending=False)
sns.barplot(data=plot_df, y="Policy", x="Max drawdown", color="#55A868")
plt.title("Aligned strict-test maximum drawdown")
plt.xlabel("Maximum drawdown")
plt.ylabel("")
plt.tight_layout()
mdd_fig = PLOT_DIR / "figure_final_max_drawdown_bar.png"
plt.savefig(mdd_fig, dpi=240)
plt.close()

# Statistical decision heatmap.
heat_df = stat_table_out.copy()
heat_pivot = heat_df.pivot_table(
    index="Metric",
    columns="Benchmark",
    values="Observed",
    aggfunc="first",
)

metric_order = [
    "Total return difference",
    "Sharpe difference",
    "Sortino difference",
    "Max drawdown improvement",
    "Calmar difference",
    "Annualized mean excess return",
]

heat_pivot = heat_pivot.reindex(metric_order)

plt.figure(figsize=(11, 5.5))
sns.heatmap(
    heat_pivot,
    annot=True,
    fmt=".3f",
    cmap="RdYlGn",
    center=0.0,
    linewidths=0.5,
)
plt.title("Observed AURORA10-UAMV-B minus benchmark differences")
plt.xlabel("Benchmark")
plt.ylabel("Metric")
plt.tight_layout()
heatmap_fig = PLOT_DIR / "figure_final_statistical_difference_heatmap.png"
plt.savefig(heatmap_fig, dpi=240)
plt.close()

# Rolling 63-day Sharpe.
rolling_fig_df = rolling_11[
    (rolling_11["policy_name"].isin(FIGURE_POLICIES))
    & (rolling_11["window"] == 63)
].copy()
rolling_fig_df["Policy"] = rolling_fig_df["policy_name"].map(policy_short_name)

plt.figure(figsize=(12, 6))
for policy, grp in rolling_fig_df.groupby("Policy"):
    grp = grp.sort_values("date")
    plt.plot(grp["date"], grp["rolling_sharpe"], linewidth=1.5, label=policy)

plt.title("Rolling 63-day annualized Sharpe")
plt.xlabel("Date")
plt.ylabel("Rolling Sharpe")
plt.grid(True, alpha=0.3)
plt.legend(fontsize=8)
plt.tight_layout()
rolling_fig = PLOT_DIR / "figure_final_rolling_63d_sharpe.png"
plt.savefig(rolling_fig, dpi=240)
plt.close()

# Copy figures to paper_figures and global figures directory.
figure_files = [
    equity_fig,
    drawdown_fig,
    sharpe_fig,
    mdd_fig,
    heatmap_fig,
    rolling_fig,
]

for src in figure_files:
    dst = PAPER_FIGURE_DIR / src.name
    dst.write_bytes(src.read_bytes())

    global_dst = FIGURE_DIR / f"{src.stem}_{RUN_ID}.png"
    global_dst.write_bytes(src.read_bytes())

figure_caption_rows = [
    {
        "Figure": "figure_final_equity_curves.png",
        "Caption": "Aligned strict-test equity curves for AURORA10-UAMV-B and primary benchmarks.",
    },
    {
        "Figure": "figure_final_drawdowns.png",
        "Caption": "Aligned strict-test drawdown curves showing the lower maximum drawdown of AURORA10-UAMV-B.",
    },
    {
        "Figure": "figure_final_sharpe_bar.png",
        "Caption": "Strict-test Sharpe ratios for final AURORA policies and primary benchmarks.",
    },
    {
        "Figure": "figure_final_max_drawdown_bar.png",
        "Caption": "Strict-test maximum drawdowns. Higher values are less negative and therefore better.",
    },
    {
        "Figure": "figure_final_statistical_difference_heatmap.png",
        "Caption": "Observed AURORA10-UAMV-B minus benchmark differences across performance and risk metrics.",
    },
    {
        "Figure": "figure_final_rolling_63d_sharpe.png",
        "Caption": "Rolling 63-day annualized Sharpe ratios for AURORA10-UAMV-B and primary benchmarks.",
    },
]

figure_caption_table = pd.DataFrame(figure_caption_rows)

save_table(
    figure_caption_table,
    local_name="paper_figure_captions.csv",
    global_table_number=110,
    description="paper_figure_captions",
)

print("Figures saved to:", PAPER_FIGURE_DIR)

# ============================================================
# 11. Manuscript text assets
# ============================================================

print("\n" + "=" * 80)
print("Step 7: Creating manuscript text assets")
print("=" * 80)

primary_perf = final_perf[final_perf["policy_name"] == PRIMARY_AURORA].iloc[0]
b6_perf = final_perf[final_perf["policy_name"] == "B6_00881_only"].iloc[0]
b3_perf = final_perf[final_perf["policy_name"] == "B3_0050_only"].iloc[0]

b6_decision = stat_table_out[stat_table_out["Benchmark"] == "00881-only"].copy()

drawdown_row = b6_decision[b6_decision["Metric"] == "Max drawdown improvement"].iloc[0]
sharpe_row = b6_decision[b6_decision["Metric"] == "Sharpe difference"].iloc[0]
return_row = b6_decision[b6_decision["Metric"] == "Total return difference"].iloc[0]

abstract_text = f"""
# Draft Abstract

This study proposes AURORA-TWETF, a leakage-controlled, uncertainty-aware regime allocation framework for Taiwan exchange-traded funds. The framework combines purged walk-forward ordinal regime forecasts with an uncertainty-aware mean-variance allocation layer. Earlier allocation variants based on hand-crafted or validation-optimized regime templates were benchmark-competitive but did not consistently outperform passive ETF baselines. The final AURORA10-UAMV-B configuration emphasized 60-day regime probabilities and increased risk aversion under forecast uncertainty and bearish conditions. On the aligned strict-test period, AURORA10-UAMV-B achieved a Sharpe ratio of {primary_perf['sharpe_ratio']:.4f}, Sortino ratio of {primary_perf['sortino_ratio']:.4f}, and maximum drawdown of {primary_perf['max_drawdown']:.4f}, compared with {b6_perf['sharpe_ratio']:.4f}, {b6_perf['sortino_ratio']:.4f}, and {b6_perf['max_drawdown']:.4f} for the strongest total-return benchmark, 00881-only. Paired circular block-bootstrap inference showed that the drawdown reduction was statistically significant at the 95% level, while Sharpe, Sortino, Calmar, and total-return differences were not. These results suggest that AURORA-TWETF is best interpreted as a downside-risk management framework rather than a return-maximizing trading system.
"""

method_text = f"""
# Methods Summary

AURORA-TWETF uses a leakage-controlled research design. Forward-return regime labels are predicted under a purged walk-forward protocol to reduce overlap-induced leakage. The final allocation layer, AURORA10-UAMV-B, combines 20-day and 60-day regime probability forecasts with weights of 0.30 and 0.70, respectively. Expected returns and covariances are estimated using trailing ETF returns only, with 63-day and 126-day lookback windows. The optimization objective is a regularized mean-variance utility with risk aversion increased by forecast uncertainty and bearish regime probability. Portfolio constraints limit individual ETF exposures, cap 00881 exposure at 30%, and allow cash allocation up to 60%. Monthly rebalancing is evaluated with 10 basis points transaction costs.
"""

results_text = f"""
# Results Summary

The final aligned strict-test comparison contains 319 trading days from 2024-11-27 to 2026-03-25. AURORA10-UAMV-B achieved total return {primary_perf['total_return']:.4f}, annual return {primary_perf['annual_return']:.4f}, annualized volatility {primary_perf['annual_volatility']:.4f}, Sharpe ratio {primary_perf['sharpe_ratio']:.4f}, Sortino ratio {primary_perf['sortino_ratio']:.4f}, and maximum drawdown {primary_perf['max_drawdown']:.4f}. The strongest total-return benchmark, 00881-only, achieved total return {b6_perf['total_return']:.4f}, annual return {b6_perf['annual_return']:.4f}, Sharpe ratio {b6_perf['sharpe_ratio']:.4f}, and maximum drawdown {b6_perf['max_drawdown']:.4f}. Thus, AURORA10-UAMV-B sacrificed total return but produced substantially lower volatility and drawdown.

Block-bootstrap inference confirmed that the maximum drawdown improvement relative to 00881-only was statistically positive at the 95% level, with observed improvement {drawdown_row['Observed']:.4f} and 95% confidence interval [{drawdown_row['CI95 lower']:.4f}, {drawdown_row['CI95 upper']:.4f}]. The observed Sharpe difference was {sharpe_row['Observed']:.4f}, but its 95% interval [{sharpe_row['CI95 lower']:.4f}, {sharpe_row['CI95 upper']:.4f}] crossed zero. The observed total-return difference was {return_row['Observed']:.4f}, and its 95% interval [{return_row['CI95 lower']:.4f}, {return_row['CI95 upper']:.4f}] also crossed zero. Therefore, the statistically supported claim is drawdown reduction rather than broad performance dominance.
"""

limitations_text = """
# Limitations

The aligned strict-test period contains only 319 trading days, so statistical inference remains sample-limited. The final UAMV configuration was identified after multiple experimental stages, which introduces model-selection risk. Although block bootstrap accounts for some serial dependence, it cannot guarantee future robustness under different market regimes. The ETF universe is also concentrated in Taiwan equity and semiconductor-related exposures, so results may not generalize to broader asset classes. Finally, the strategy sacrifices total return relative to high-beta passive exposure, making it more appropriate for downside-risk management than for return maximization.
"""

conclusion_text = """
# Conclusion

AURORA-TWETF provides a reproducible uncertainty-aware allocation framework for Taiwan ETF portfolios. The experiments show that regime prediction alone is insufficient; the allocation layer determines whether probabilistic forecasts translate into portfolio value. Hand-crafted and validation-optimized regime templates did not outperform strong passive baselines. In contrast, the uncertainty-aware mean-variance layer produced materially lower downside risk. Statistical testing supports significant drawdown reduction, while Sharpe and total-return improvements were not statistically significant. The final contribution is therefore best framed as a leakage-controlled, uncertainty-aware downside-risk management framework rather than a trading system that reliably maximizes return.
"""

caption_text = """
# Paper Figure Captions

Figure 1. Aligned strict-test equity curves for AURORA10-UAMV-B and primary benchmarks.

Figure 2. Aligned strict-test drawdown curves. AURORA10-UAMV-B shows materially lower maximum drawdown than passive ETF benchmarks.

Figure 3. Strict-test Sharpe ratios for AURORA10-UAMV policies and primary benchmarks.

Figure 4. Strict-test maximum drawdowns. Less negative values indicate better downside-risk control.

Figure 5. Observed AURORA10-UAMV-B minus benchmark differences across return, risk-adjusted, and drawdown metrics.

Figure 6. Rolling 63-day annualized Sharpe ratios for AURORA10-UAMV-B and primary benchmarks.
"""

claim_checklist_text = """
# Claim Checklist

Supported claims:
- AURORA10-UAMV-B achieved the best aligned strict-test composite rank among the tested policies.
- AURORA10-UAMV-B reduced maximum drawdown relative to all primary benchmarks.
- The drawdown improvement was statistically significant at the 95% block-bootstrap level.
- AURORA10-UAMV-B improved observed Sharpe and Sortino ratios relative to primary benchmarks.

Unsupported or unsafe claims:
- Do not claim statistically significant Sharpe improvement.
- Do not claim statistically significant Sortino improvement.
- Do not claim total-return superiority.
- Do not claim future outperformance.
- Do not describe the method as personalized financial advice.
"""

write_markdown(MANUSCRIPT_DIR / "draft_abstract.md", abstract_text)
write_markdown(MANUSCRIPT_DIR / "methods_summary.md", method_text)
write_markdown(MANUSCRIPT_DIR / "results_summary.md", results_text)
write_markdown(MANUSCRIPT_DIR / "limitations.md", limitations_text)
write_markdown(MANUSCRIPT_DIR / "conclusion.md", conclusion_text)
write_markdown(MANUSCRIPT_DIR / "figure_captions.md", caption_text)
write_markdown(MANUSCRIPT_DIR / "claim_checklist.md", claim_checklist_text)

combined_manuscript_assets = "\n\n".join([
    abstract_text,
    method_text,
    results_text,
    limitations_text,
    conclusion_text,
    caption_text,
    claim_checklist_text,
])

write_markdown(MANUSCRIPT_DIR / "AURORA_TWETF_manuscript_assets_combined.md", combined_manuscript_assets)

# ============================================================
# 12. Final evidence summary table
# ============================================================

print("\n" + "=" * 80)
print("Step 8: Creating final evidence summary")
print("=" * 80)

evidence_rows = [
    {
        "Evidence item": "Best final allocation method",
        "Finding": "AURORA10-UAMV-B",
        "Support": "Best aligned strict-test composite rank in Notebook 10.",
        "Paper implication": "Use as primary AURORA method.",
    },
    {
        "Evidence item": "Total return",
        "Finding": "AURORA10-UAMV-B underperformed 00881-only in total return.",
        "Support": f"AURORA total return {primary_perf['total_return']:.4f}; 00881-only {b6_perf['total_return']:.4f}.",
        "Paper implication": "Do not claim return superiority.",
    },
    {
        "Evidence item": "Sharpe ratio",
        "Finding": "Observed Sharpe improvement, not statistically significant.",
        "Support": f"Observed Sharpe difference vs 00881-only {sharpe_row['Observed']:.4f}; CI crosses zero.",
        "Paper implication": "Describe as observed improvement only.",
    },
    {
        "Evidence item": "Sortino ratio",
        "Finding": "Observed Sortino improvement, not statistically significant.",
        "Support": "Notebook 11 block-bootstrap interval crosses zero.",
        "Paper implication": "Describe as observed improvement only.",
    },
    {
        "Evidence item": "Maximum drawdown",
        "Finding": "Statistically significant drawdown reduction.",
        "Support": f"Drawdown improvement vs 00881-only {drawdown_row['Observed']:.4f}; 95% CI [{drawdown_row['CI95 lower']:.4f}, {drawdown_row['CI95 upper']:.4f}].",
        "Paper implication": "Primary supported contribution.",
    },
    {
        "Evidence item": "Validation-optimized templates",
        "Finding": "Underperformed benchmarks.",
        "Support": "Notebook 09 comparison rankings.",
        "Paper implication": "Use as ablation showing allocation-layer overfitting.",
    },
    {
        "Evidence item": "Hand-crafted templates",
        "Finding": "Benchmark-competitive but not dominant.",
        "Support": "Notebook 08B aligned evaluation.",
        "Paper implication": "Use as baseline allocation layer.",
    },
]

evidence_table = pd.DataFrame(evidence_rows)

save_table(
    evidence_table,
    local_name="paper_final_evidence_summary.csv",
    global_table_number=111,
    description="paper_final_evidence_summary",
)

save_latex_table(
    evidence_table,
    filename="table_final_evidence_summary.tex",
    caption="Final evidence summary and paper claim boundaries.",
    label="tab:evidence_summary",
)

print(evidence_table.to_string(index=False))

# ============================================================
# 13. Output index
# ============================================================

print("\n" + "=" * 80)
print("Step 9: Creating output index")
print("=" * 80)

output_index_rows = []

for path in sorted(TABLE_RUN_DIR.glob("*.csv")):
    output_index_rows.append({
        "asset_type": "table_csv",
        "path": str(path),
        "description": path.stem,
    })

for path in sorted(LATEX_DIR.glob("*.tex")):
    output_index_rows.append({
        "asset_type": "latex_table",
        "path": str(path),
        "description": path.stem,
    })

for path in sorted(PAPER_FIGURE_DIR.glob("*.png")):
    output_index_rows.append({
        "asset_type": "paper_figure",
        "path": str(path),
        "description": path.stem,
    })

for path in sorted(MANUSCRIPT_DIR.glob("*.md")):
    output_index_rows.append({
        "asset_type": "manuscript_text",
        "path": str(path),
        "description": path.stem,
    })

output_index_df = pd.DataFrame(output_index_rows)

output_index_df.to_csv(RUN_ROOT / "NOTEBOOK12_OUTPUT_INDEX.csv", index=False)
output_index_df.to_csv(TABLE_DIR / f"table_112_notebook12_output_index_{RUN_ID}.csv", index=False)

print(output_index_df.to_string(index=False))

# ============================================================
# 14. Validation report and manifest
# ============================================================

print("\n" + "=" * 80)
print("Step 10: Saving validation report and manifest")
print("=" * 80)

validation_report = {
    "project_code": PROJECT_CODE,
    "notebook": "12_AURORA_paper_tables_figures_and_manuscript_assets.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "notebook08b_run_id": NOTEBOOK08B_RUN_ID,
    "notebook09_run_id": NOTEBOOK09_RUN_ID,
    "notebook10_run_id": NOTEBOOK10_RUN_ID,
    "notebook11_run_id": NOTEBOOK11_RUN_ID,
    "primary_aurora_policy": PRIMARY_AURORA,
    "primary_benchmarks": PRIMARY_BENCHMARKS,
    "main_supported_claim": (
        "AURORA10-UAMV-B significantly reduced maximum drawdown relative to primary benchmarks "
        "under paired circular block-bootstrap inference."
    ),
    "unsupported_claims": [
        "Statistically significant Sharpe superiority",
        "Statistically significant Sortino superiority",
        "Total-return superiority",
        "Future outperformance",
        "Personalized financial advice",
    ],
    "key_results": {
        "aurora_total_return": float(primary_perf["total_return"]),
        "aurora_sharpe": float(primary_perf["sharpe_ratio"]),
        "aurora_sortino": float(primary_perf["sortino_ratio"]),
        "aurora_max_drawdown": float(primary_perf["max_drawdown"]),
        "b6_total_return": float(b6_perf["total_return"]),
        "b6_sharpe": float(b6_perf["sharpe_ratio"]),
        "b6_sortino": float(b6_perf["sortino_ratio"]),
        "b6_max_drawdown": float(b6_perf["max_drawdown"]),
        "drawdown_improvement_vs_b6": float(drawdown_row["Observed"]),
        "drawdown_improvement_ci95_lower_vs_b6": float(drawdown_row["CI95 lower"]),
        "drawdown_improvement_ci95_upper_vs_b6": float(drawdown_row["CI95 upper"]),
    },
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "tables": str(TABLE_RUN_DIR),
        "latex_tables": str(LATEX_DIR),
        "plots": str(PLOT_DIR),
        "paper_figures": str(PAPER_FIGURE_DIR),
        "manuscript_assets": str(MANUSCRIPT_DIR),
        "reports": str(REPORT_RUN_DIR),
    },
    "educational_note": (
        "This notebook prepares research-paper assets only and does not provide personalized financial advice."
    ),
}

validation_report_path = REPORT_RUN_DIR / "AURORA_12_paper_assets_validation_report.json"
validation_report_global_path = REPORT_DIR / f"AURORA_12_paper_assets_validation_report_{RUN_ID}.json"

save_json(validation_report_path, validation_report)
save_json(validation_report_global_path, validation_report)

manifest_df = make_file_manifest(RUN_ROOT)

manifest_path = REPORT_RUN_DIR / "AURORA_12_file_manifest_SHA256.csv"
manifest_global_path = REPORT_DIR / f"AURORA_12_file_manifest_SHA256_{RUN_ID}.csv"

manifest_df.to_csv(manifest_path, index=False)
manifest_df.to_csv(manifest_global_path, index=False)

# ============================================================
# 15. Final summary
# ============================================================

print("\n" + "=" * 80)
print("AURORA-TWETF NOTEBOOK 12 COMPLETE")
print("=" * 80)
print("Run ID                         :", RUN_ID)
print("Run root                       :", RUN_ROOT)
print("Final performance table        :", TABLE_DIR / f"table_106_paper_final_performance_table_{RUN_ID}.csv")
print("Statistical inference table    :", TABLE_DIR / f"table_107_paper_statistical_inference_table_{RUN_ID}.csv")
print("Methodology progression table  :", TABLE_DIR / f"table_108_paper_methodology_progression_table_{RUN_ID}.csv")
print("UAMV-B configuration table     :", TABLE_DIR / f"table_109_paper_aurora10_uamv_b_configuration_{RUN_ID}.csv")
print("Figure captions table          :", TABLE_DIR / f"table_110_paper_figure_captions_{RUN_ID}.csv")
print("Final evidence summary         :", TABLE_DIR / f"table_111_paper_final_evidence_summary_{RUN_ID}.csv")
print("Notebook 12 output index       :", TABLE_DIR / f"table_112_notebook12_output_index_{RUN_ID}.csv")
print("Latex tables                   :", LATEX_DIR)
print("Paper figures                  :", PAPER_FIGURE_DIR)
print("Manuscript assets              :", MANUSCRIPT_DIR)
print("Validation report              :", validation_report_path)
print("Manifest                       :", manifest_path)
print("=" * 80)

print("\nRecommended next step:")
print("Use the manuscript assets to draft the IEEE-style paper sections.")
print("Primary supported claim: statistically significant drawdown reduction, not total-return superiority.")

Mounted at /content/drive
AURORA-TWETF Notebook 12: Paper Tables, Figures, and Manuscript Assets
Timestamp UTC : 2026-06-24T14:23:07Z
Run ID        : 20260624_142307
Notebook 08B  : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/aligned_oos_reanalysis/run_20260624_070827
Notebook 09   : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/validation_optimized_regime_templates/run_20260624_072914
Notebook 10   : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/uncertainty_aware_mean_variance_allocation/run_20260624_100748
Notebook 11   : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/statistical_significance_block_bootstrap/run_20260624_124834
Run root      : /content/drive/MyDrive/AURORA_TWETF/outputs/AURORA_TWETF/paper_tables_figures_manuscript_assets/run_20260624_142307

Step 1: Loading result tables
Notebook 08B aligned test rankings: (23, 28)
Notebook 09 optimized rankings     : (23, 26)
Notebook 10 UAMV rankings          : (25, 27)
Notebook 11 decis